In [1]:
# Google Colab Only
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
from collections.abc import Iterator
from typing import cast

import numpy as np
import polars as pl
from IPython.display import display
from maite_datasets.multiobject_tracking import (
    MultiobjectTrackingTargetTuple,
    SingleFrameObjectTrackingTargetTuple,
    VideoFrameTuple,
)

from dataeval.config import set_max_processes
from dataeval.data import FrameIndices, SequenceFrames
from dataeval.protocols import (
    DatasetMetadata,
    DatumMetadata,
    MultiobjectTrackingDatum,
    VideoFrame,
)
from dataeval.quality import Duplicates

set_max_processes(4)
pl.Config.set_tbl_width_chars(160)

polars.config.Config

In [3]:
HEIGHT, WIDTH = 72, 96
YY, XX = np.mgrid[0:HEIGHT, 0:WIDTH]


def scene(seed: int, n_frames: int) -> list[np.ndarray]:
    """A textured ground plane with two objects tracking across it."""
    rng = np.random.default_rng(seed)
    terrain = rng.normal(70, 12, (HEIGHT, WIDTH))
    for _ in range(5):
        cx, cy, spread = rng.uniform(0, WIDTH), rng.uniform(0, HEIGHT), rng.uniform(150, 900)
        terrain += rng.uniform(-90, 110) * np.exp(-(((XX - cx) ** 2 + (YY - cy) ** 2) / spread))
    terrain[: rng.integers(HEIGHT // 4, 3 * HEIGHT // 4)] += rng.uniform(30, 70)

    frames = []
    for i in range(n_frames):
        image = terrain.copy()
        for k, (speed, y0, size) in enumerate(((1.4, 0.30, 90.0), (0.9, 0.68, 60.0))):
            cx = (6 + i * speed * 1.6) % (WIDTH - 12) + 6
            image += (150 - 40 * k) * np.exp(-(((XX - cx) ** 2 + (YY - HEIGHT * y0) ** 2) / size))
        frames.append(np.clip(image, 0, 255).astype(np.uint8))
    return frames


def transcode(frames: list[np.ndarray], seed: int) -> list[np.ndarray]:
    """The same footage through another codec: same content, different bytes."""
    rng = np.random.default_rng(seed)
    return [np.clip(f.astype(np.float64) + rng.normal(0, 6, f.shape), 0, 255).astype(np.uint8) for f in frames]

In [4]:
def boxes_at(frame_index: int) -> SingleFrameObjectTrackingTargetTuple:
    """Two tracked objects following the same paths the pixels do."""
    corners, tracks = [], []
    for track, speed, y0 in ((0, 1.4, 0.30), (1, 0.9, 0.68)):
        cx = (6 + frame_index * speed * 1.6) % (WIDTH - 12) + 6
        corners.append([cx - 9, HEIGHT * y0 - 9, cx + 9, HEIGHT * y0 + 9])
        tracks.append(track)
    return SingleFrameObjectTrackingTargetTuple(
        boxes=np.array(corners, dtype=np.float32),
        labels=np.array([0, 1], dtype=np.int64),
        scores=np.ones(2, dtype=np.float32),
        track_ids=np.array(tracks, dtype=np.int64),
    )


class VideoStream:
    """An iterable of decoded frames, standing in for a file a decoder would walk."""

    def __init__(self, frames: list[np.ndarray], fps: float = 30.0) -> None:
        self._frames, self._fps = frames, fps

    def __iter__(self) -> Iterator[VideoFrame]:
        for index, frame in enumerate(self._frames):
            pixels = np.stack([frame, frame, frame])
            yield VideoFrameTuple(pixels=pixels, time_s=index / self._fps, pts=index, frame_index=index)


class VideoDataset:
    """A minimal MAITE multi-object tracking dataset over in-memory footage."""

    def __init__(self, sequences: dict[str, list[np.ndarray]], dataset_id: str) -> None:
        self.metadata = DatasetMetadata({"id": dataset_id, "index2label": {0: "vehicle", 1: "person"}})
        self._data: list[MultiobjectTrackingDatum] = [
            (
                VideoStream(frames),
                MultiobjectTrackingTargetTuple(frame_tracks=[boxes_at(i) for i in range(len(frames))]),
                cast(DatumMetadata, {"id": name, "height": HEIGHT, "width": WIDTH}),
            )
            for name, frames in sequences.items()
        ]

    def __len__(self) -> int:
        return len(self._data)

    def __getitem__(self, index: int) -> MultiobjectTrackingDatum:
        return self._data[index]

In [5]:
alpha = scene(1, 60)
bravo = scene(5, 40)
bravo_stare = bravo[:15] + [bravo[15]] * 20 + bravo[16:]

train = VideoDataset(
    {
        "patrol_alpha": alpha,
        "patrol_alpha_transcode": transcode(alpha, seed=9),
        "patrol_bravo": bravo_stare,
        "patrol_charlie": scene(12, 45),
    },
    dataset_id="train",
)
test = VideoDataset({"eval_clip": transcode(alpha[30:50], seed=3)}, dataset_id="test")

In [6]:
strict = Duplicates().evaluate(train)
relaxed = Duplicates(hash_radius=6).evaluate(train)

for name, result in (("hash_radius=0", strict), ("hash_radius=6", relaxed)):
    row = result.sequences.data().row(0, named=True)
    print(
        f"{name}: sequences {row['item_indices']} share frames "
        f"{row['span_start'][0]}-{row['span_end'][0]}, containment {[round(c, 2) for c in row['containment']]}"
    )

hash_radius=0: sequences [0, 1] share frames 1-57, containment [0.52, 0.52]
hash_radius=6: sequences [0, 1] share frames 0-59, containment [1.0, 1.0]


In [7]:
for name, radius in (("hash_radius=0", 0), ("hash_radius=6", 6)):
    found = Duplicates(hash_radius=radius, min_segment_frames=10).evaluate(train, test)
    row = found.sequences.data().filter(pl.col("dataset_indices").list.n_unique() > 1).row(0, named=True)
    print(
        f"{name}: train frames {row['span_start'][0]}-{row['span_end'][0]} "
        f"== test frames {row['span_start'][1]}-{row['span_end'][1]}"
        f"   ({row['containment'][1]:.0%} of the test clip)"
    )

hash_radius=0: train frames 39-48 == test frames 9-18   (60% of the test clip)


hash_radius=6: train frames 30-49 == test frames 0-19   (100% of the test clip)


In [8]:
summary = relaxed.aggregate_by_sequence()
display(summary)

sequence,n_frames,redundant_frames,redundant_fraction,longest_run,duplicate_frames,shared_with,group_count
i64,u32,u32,f64,u32,u32,u32,u32
0,60,37,0.616667,9,60,1,18
1,60,33,0.55,6,60,1,21
2,59,46,0.779661,20,0,0,14
3,45,23,0.511111,5,0,0,13


In [9]:
display(
    relaxed.sequences.data().select(
        "dup_type", "item_indices", "span_start", "span_end", "containment", "mean_distance"
    )
)

dup_type,item_indices,span_start,span_end,containment,mean_distance
str,list[i64],list[i64],list[i64],list[f64],f64
"""segment""","[0, 1]","[0, 0]","[59, 59]","[1.0, 1.0]",1.033333


In [10]:
leakage = Duplicates(hash_radius=6, min_segment_frames=10).evaluate(train, test)
leaks = leakage.crossing.aggregate_by_pair("sequence")
display(leaks)

level,dataset_a,dataset_b,item_a,item_b,relations,n_groups,containment_a,containment_b,mean_distance
str,i64,i64,i64,i64,list[str],u32,f64,f64,f64
"""sequence""",0,1,0,0,"[""segment""]",1,0.4,1.0,1.0
"""sequence""",0,1,1,0,"[""segment""]",1,0.416667,1.0,1.3


In [11]:
for row in leakage.sequences.data().filter(pl.col("dataset_indices").list.n_unique() > 1).iter_rows(named=True):
    train_seq, test_seq = row["item_indices"]
    print(
        f"train[{train_seq}] frames {row['span_start'][0]}-{row['span_end'][0]}"
        f"  ==  test[{test_seq}] frames {row['span_start'][1]}-{row['span_end'][1]}"
        f"   ({row['containment'][1]:.0%} of the test clip)"
    )

train[0] frames 30-49  ==  test[0] frames 0-19   (100% of the test clip)
train[1] frames 30-49  ==  test[0] frames 0-19   (100% of the test clip)


In [12]:
runs = (
    relaxed
    .data()
    .filter(pl.col("dup_type") == "redundant")
    .with_columns(pl.col("unit_indices").list.len().alias("run_length"))
    .sort("run_length", descending=True)
    .select("item_indices", "unit_indices", "run_length", "mean_distance")
)
display(runs.head(4))

item_indices,unit_indices,run_length,mean_distance
list[i64],list[i64],u32,f64
"[2, 2, … 2]","[15, 16, … 34]",20,0.0
"[0, 0, … 0]","[25, 26, … 33]",9,3.5
"[2, 2, … 2]","[0, 1, … 6]",7,2.666667
"[1, 1, … 1]","[25, 26, … 30]",6,3.6


In [13]:
display(summary.select("sequence", "redundant_fraction", "longest_run"))
#
# A static run of *k* frames can be reduced to a single frame without a loss of semantic content. The `redundant_frames`
# metric quantifies this potential reduction. However, you should evaluate these frames carefully before removing them:
# temporal dwell can carry valuable signal, and an object tracker trained exclusively on moving targets may fail to
# detect stationary objects.

sequence,redundant_fraction,longest_run
i64,f64,u32
0,0.616667,9
1,0.55,6
2,0.779661,20
3,0.511111,5


In [14]:
source = scene(1, 40)
slowed = [frame for frame in source for _ in (0, 1)]
repackaged = VideoDataset({"source": source, "slowed_export": transcode(slowed, seed=4)}, dataset_id="repack")

without = Duplicates(hash_radius=6, min_segment_frames=20).evaluate(repackaged)
print(f"segments only:              {without.sequences.data().shape[0]} relation(s) found")

warped = Duplicates(hash_radius=6, min_segment_frames=20, verify_alignment=8).evaluate(repackaged)
print(f"with verify_alignment=8:    {warped.sequences.data().shape[0]} relation(s) found")

segments only:              0 relation(s) found


with verify_alignment=8:    1 relation(s) found


In [15]:
display(
    warped.sequences.data().select("dup_type", "item_indices", "span_start", "span_end", "containment", "mean_distance")
)

dup_type,item_indices,span_start,span_end,containment,mean_distance
str,list[i64],list[i64],list[i64],list[f64],f64
"""aligned""","[0, 1]","[0, 3]","[39, 74]","[1.0, 0.9]",1.173333


In [16]:
tracked = Duplicates(hash_radius=6, min_track_frames=10).evaluate(train, levels="track")
display(
    tracked.tracks.data().select("dup_type", "item_indices", "track_indices", "span_start", "span_end", "containment")
)

dup_type,item_indices,track_indices,span_start,span_end,containment
str,list[i64],list[i64],list[i64],list[i64],list[f64]
"""segment""","[0, 1]","[0, 0]","[1, 1]","[59, 59]","[0.583333, 0.6]"
"""segment""","[0, 1]","[1, 1]","[1, 1]","[40, 40]","[0.566667, 0.566667]"
"""segment""","[0, 1]","[1, 1]","[48, 48]","[58, 58]","[0.566667, 0.566667]"


In [17]:
row = leakage.crossing.sequences.data().row(0, named=True)
sequence = row["item_indices"][0]
first, last = row["span_start"][0], row["span_end"][0]

leaked = SequenceFrames(train, FrameIndices({sequence: range(first, last + 1)}))
print(f"the leaked stretch is {len(leaked)} frames of sequence {sequence}")

pixels, target, metadata = next(iter(leaked.stream()))
# The per-frame keys DataEval adds live alongside the ones MAITE declares, so read them as a dict.
meta = dict(metadata)
print(f"first of them: frame {meta['frame']} of sequence {meta['sequence']}, pixels {pixels.shape}")

the leaked stretch is 20 frames of sequence 0
first of them: frame 30 of sequence 0, pixels (3, 72, 96)


In [18]:
n_frames = len(train[sequence][1].frame_tracks)
keep = [frame for frame in range(n_frames) if not first <= frame <= last]
without_leak = SequenceFrames(train, FrameIndices({sequence: keep}))
print(f"sequence {sequence} without the shared stretch: {len(without_leak)} of {n_frames} frames")

sequence 0 without the shared stretch: 40 of 60 frames


In [19]:
# TEST ASSERTION CELL ###
# hash_radius matters: the strict default under-reports the relation rather than missing it
assert strict.sequences.data().row(0, named=True)["containment"][0] < 0.6
assert relaxed.sequences.data().shape[0] == 1

# one relation, not one per diagonal
alpha_row = relaxed.sequences.data().row(0, named=True)
assert alpha_row["item_indices"] == [0, 1]
assert alpha_row["containment"] == [1.0, 1.0]

# self-repetition is not cross-sequence duplication
assert summary["duplicate_frames"].to_list() == [60, 60, 0, 0]
assert summary["shared_with"].to_list() == [1, 1, 0, 0]
assert summary["redundant_fraction"].to_list()[2] == max(summary["redundant_fraction"].to_list())

# leakage: the test clip is entirely drawn from training footage
assert leaks.shape[0] >= 1
first = leaks.row(0, named=True)
assert (first["dataset_a"], first["dataset_b"]) == (0, 1)
assert first["containment_b"] == 1.0
assert first["containment_a"] < 0.5
spans = leakage.sequences.data().filter(pl.col("dataset_indices").list.n_unique() > 1).row(0, named=True)
assert (spans["span_start"][0], spans["span_end"][0]) == (30, 49)
assert (spans["span_start"][1], spans["span_end"][1]) == (0, 19)

# the stare is the longest redundant run, and it is exact
assert runs.row(0, named=True)["run_length"] == 20
assert runs.row(0, named=True)["mean_distance"] == 0.0

# warping finds what a fixed offset cannot
assert without.sequences.data().shape[0] == 0
assert warped.sequences.data().shape[0] == 1
assert warped.sequences.data().row(0, named=True)["dup_type"] == "aligned"

# a finding leads back to the frames it names
assert len(leaked) == 20
assert meta["frame"] == 30
assert meta["sequence"] == 0
assert len(without_leak) == 40

# tracks travelled with the copied clip
assert tracked.tracks.data().shape[0] > 0
assert all(row == [0, 1] for row in tracked.tracks.data()["item_indices"].to_list())
# a level not asked for is not searched for
assert set(tracked.data()["level"].to_list()) == {"track"}